# Minimum information tiers: gene clusters and co-expression

Do genes that are co-expressed (here: in the same MEANtools functional cluster) lie in the same biosynthetic gene cluster, and do they match PlantMetWiki pathways?

**Case:** cases 3 and 4: plantiSMASH-predicted BGCs and gene co-expression (with case 6). **Data:** association, sample (evidence graphs of the pilots, `shape_constraints/rdf/` and `examples/`). **Queries:** the files of [`sparql-examples-pmw`](https://github.com/pathway-lod/sparql-examples-pmw/tree/main/examples/MinimumInformationTiers) (<https://pathway-lod.github.io/sparql-examples-pmw/>), run in a local Oxigraph store, with PlantMetWiki, Rhea and UniProt called by `SERVICE` on the live endpoints.

The outputs are a **dated snapshot**: the date and the PlantMetWiki release are in the summary table at the end. Each query is run once with a time limit; a tier that did not finish is shown as *timeout*. The results of every tier are also saved as CSV in `results/`. To rerun, open the notebook with a kernel that has the packages of `requirements.txt` installed (see the README of this folder).

In [1]:
import mi
import pandas as pd
from IPython.display import display
pd.set_option('display.max_colwidth', 80)
CASE = '3 gene clusters'
print('PlantMetWiki release:', mi.pmw_release())

PlantMetWiki release: r17.0.0_20260605-171052


## The data

In [2]:
data = ['association', 'sample']
print('gene-metabolite associations:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> SELECT (COUNT(?a) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation }'))
print('MEANtools functional clusters:', mi.local_count(data, 'PREFIX lodp: <https://example.org/lod-pathways/pilot/> PREFIX sio: <http://semanticscience.org/resource/> SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?a a lodp:GeneMetaboliteAssociation ; sio:SIO_000772/lodp:functionalCluster ?c }'))
print('biosynthetic gene clusters in the sample graph:', mi.local_count(data, 'PREFIX obo: <http://purl.obolibrary.org/obo/> SELECT (COUNT(DISTINCT ?c) AS ?n) WHERE { ?c a obo:SO_0002357 }'))

gene-metabolite associations: 1294
MEANtools functional clusters: 53


biosynthetic gene clusters in the sample graph: 1


## Co-expressed genes are in the same biosynthetic gene cluster

`coexpressed-in-same-cluster`

Two genes that are co-expressed (here: genes associated with the same metabolite feature in the same MEANtools functional cluster) are both parts of the same plantiSMASH biosynthetic gene cluster (cluster obo:BFO_0000051 gene). Returns the pairs and the cluster. NOT RUNNABLE ON THE CURRENT GRAPHS: the cluster graph lists only the genes linked to a pathway, not all genes of the cluster, and the tomato clusters are not in the RDF.

*Keywords:* MEANtools, Minimum Information: evidence tiers, biosynthetic gene cluster, case 3: plantiSMASH-predicted BGCs, predicted gene clusters, case 4: transcriptomics, gene co-expression values, case 6: paired omics, transcript-metabolite associations, co-expression, evidence tier, gene, gene-metabolite association, local evidence graph, no data yet, plantiSMASH

In [3]:
r = mi.run('coexpressed-in-same-cluster', ['association', 'sample'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 2.0 s


,cluster,geneA,geneB,functionalCluster


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Same cluster, only one of the genes matches a pathway

`coexpressed-in-cluster-one-in-pathway`

As in `coexpressed-in-same-cluster`, and only one of the two genes is listed in a PlantMetWiki pathway (by its Solyc locus). This is a lead to extend that pathway with the other gene. NOT RUNNABLE ON THE CURRENT GRAPHS (see `coexpressed-in-same-cluster`). TESTED 2026-11-10: 0 rows in 149 s on the association graph and the sample graph.

*Keywords:* Minimum Information: evidence tiers, PlantMetWiki pathway, Solyc locus, biosynthetic gene cluster, case 3: plantiSMASH-predicted BGCs, predicted gene clusters, case 4: transcriptomics, gene co-expression values, case 6: paired omics, transcript-metabolite associations, co-expression, evidence tier, federated query, gene, local evidence graph, no data yet, pathway expansion, plantiSMASH

In [4]:
r = mi.run('coexpressed-in-cluster-one-in-pathway', ['association', 'sample'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

no rows | 0 rows | 85.5 s


,cluster,geneInPathway,geneNotInPathway,pathway


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Same cluster, both genes match a pathway

`coexpressed-in-cluster-both-in-pathway`

As in `coexpressed-in-same-cluster`, and both genes are listed in a PlantMetWiki pathway (by their Solyc loci); returns the pathways of the two genes and whether it is the same pathway. NOT RUNNABLE ON THE CURRENT GRAPHS (see `coexpressed-in-same-cluster`). TESTED 2026-11-10: did NOT finish within 280 s (two unbound federated calls to PlantMetWiki); needs a LIMIT or a cached lookup.

*Keywords:* Minimum Information: evidence tiers, PlantMetWiki pathway, Solyc locus, biosynthetic gene cluster, case 3: plantiSMASH-predicted BGCs, predicted gene clusters, case 4: transcriptomics, gene co-expression values, case 6: paired omics, transcript-metabolite associations, co-expression, evidence tier, federated query, gene, local evidence graph, no data yet, plantiSMASH

In [5]:
r = mi.run('coexpressed-in-cluster-both-in-pathway', ['association', 'sample'], CASE, timeout=150)
mi.record(r)
print(r['status'], '|', r['n_rows'], 'rows |', r['seconds'], 's')
display(r['table'].head(10))

timeout | None rows | None s


""


**For discussion:** is this the right rule for the tier? Is the number of results what we expect? What would make it more useful (a threshold, another data source, a different match)?

*Comments:* 

## Summary of this case

In [6]:
s = pd.read_csv('results/summary.csv')
display(s[s['case'] == CASE].reset_index(drop=True))

,tier,case,status,n_rows,seconds,timeout_s,date,pmw_release
0,coexpressed-in-cluster-both-in-pathway,3 gene clusters,timeout,NaN,NaN,150,2026-11-10,r17.0.0_20260605-171052
1,coexpressed-in-cluster-one-in-pathway,3 gene clusters,no rows,0.0,85.5,150,2026-11-10,r17.0.0_20260605-171052
2,coexpressed-in-same-cluster,3 gene clusters,no rows,0.0,2.0,150,2026-11-10,r17.0.0_20260605-171052
